## Setup Folder Project

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import re
import sys
import subprocess
import zipfile
from urllib.parse import urlparse

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_raw = direktori_project / "data" / "raw"
direktori_processed = direktori_project / "data" / "processed"
direktori_outputs = direktori_project / "reports" / "outputs"

direktori_phreshphish = direktori_raw / "phreshphish"
direktori_deepurlbench = direktori_raw / "deepurlbench"
direktori_tranco = direktori_raw / "tranco"
direktori_multi = direktori_processed / "multi_dataset"

for folder in [
    direktori_raw,
    direktori_processed,
    direktori_outputs,
    direktori_phreshphish,
    direktori_deepurlbench,
    direktori_tranco,
    direktori_multi,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder raw:", direktori_raw)
print("Folder PhreshPhish:", direktori_phreshphish)
print("Folder DeepURLBench:", direktori_deepurlbench)
print("Folder Tranco:", direktori_tranco)
print("Folder multi dataset:", direktori_multi)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder raw: C:\Users\ASUS\PHISHING\data\raw
Folder PhreshPhish: C:\Users\ASUS\PHISHING\data\raw\phreshphish
Folder DeepURLBench: C:\Users\ASUS\PHISHING\data\raw\deepurlbench
Folder Tranco: C:\Users\ASUS\PHISHING\data\raw\tranco
Folder multi dataset: C:\Users\ASUS\PHISHING\data\processed\multi_dataset


## Install Package yang Dibutuhkan

In [2]:
package_dibutuhkan = [
    "datasets",
    "huggingface_hub",
    "pyarrow",
    "fastparquet",
    "tqdm",
]

for package in package_dibutuhkan:
    try:
        __import__(package)
        print(f"Package sudah tersedia: {package}")
    except ImportError:
        print(f"Package belum tersedia, install: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("Selesai cek package.")

Package sudah tersedia: datasets
Package sudah tersedia: huggingface_hub
Package sudah tersedia: pyarrow
Package sudah tersedia: fastparquet
Package sudah tersedia: tqdm
Selesai cek package.


## Helper Standardisasi Dataset

In [3]:
from tqdm.auto import tqdm


def bersihkan_url(url):
    if pd.isna(url):
        return ""

    url = str(url).strip()
    url = url.replace("\x00", "")
    url = re.sub(r"\s+", "", url)

    return url


def ambil_domain_dari_url(url):
    url = bersihkan_url(url)

    if not url:
        return ""

    try:
        if not re.match(r"^https?://", url, flags=re.I):
            url_parse = "http://" + url
        else:
            url_parse = url

        parsed = urlparse(url_parse)
        domain = parsed.netloc.lower()
        domain = domain.split("@")[-1]
        domain = domain.split(":")[0]
        domain = domain.replace("www.", "", 1)

        return domain
    except Exception:
        return ""


def normalisasi_label_ke_target(label):
    teks = str(label).strip().lower()

    label_aman = {
        "0",
        "benign",
        "legit",
        "legitimate",
        "safe",
        "good",
        "normal",
        "clean",
        "tranco_legitimate",
    }

    label_berisiko = {
        "1",
        "phish",
        "phishing",
        "mal",
        "malware",
        "malicious",
        "bad",
        "defacement",
        "suspicious",
        "threat",
    }

    if teks in label_aman:
        return 0

    if teks in label_berisiko:
        return 1

    return np.nan


def cari_kolom(data, kandidat):
    kolom_lower = {kolom.lower(): kolom for kolom in data.columns}

    for nama in kandidat:
        if nama.lower() in kolom_lower:
            return kolom_lower[nama.lower()]

    return None


def buat_dataset_standar(data, nama_dataset, sumber_data, split="unknown"):
    data = data.copy()

    kolom_url = cari_kolom(data, ["url", "URL", "Url", "link", "Link", "domain", "Domain"])
    kolom_label = cari_kolom(data, ["label", "Label", "target", "class", "type", "status", "original_label"])

    if kolom_url is None:
        raise ValueError(f"Kolom URL tidak ditemukan. Kolom tersedia: {list(data.columns)}")

    if kolom_label is None:
        raise ValueError(f"Kolom label tidak ditemukan. Kolom tersedia: {list(data.columns)}")

    hasil = pd.DataFrame()
    hasil["url"] = data[kolom_url].apply(bersihkan_url)
    hasil["domain"] = hasil["url"].apply(ambil_domain_dari_url)
    hasil["original_label"] = data[kolom_label].astype(str)
    hasil["target_phishing"] = hasil["original_label"].apply(normalisasi_label_ke_target)
    hasil["dataset_name"] = nama_dataset
    hasil["sumber_data"] = sumber_data
    hasil["split"] = split
    hasil["tanggal_diproses"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if "first_seen" in data.columns:
        hasil["first_seen"] = data["first_seen"].astype(str)

    hasil = hasil[hasil["url"] != ""]
    hasil = hasil[hasil["domain"] != ""]
    hasil = hasil.dropna(subset=["target_phishing"])
    hasil["target_phishing"] = hasil["target_phishing"].astype(int)
    hasil = hasil.drop_duplicates(subset=["url"]).reset_index(drop=True)

    return hasil


def simpan_ringkasan_dataset(data, nama_file):
    ringkasan = {
        "jumlah_data": int(len(data)),
        "jumlah_aman": int((data["target_phishing"] == 0).sum()) if "target_phishing" in data.columns else 0,
        "jumlah_berisiko": int((data["target_phishing"] == 1).sum()) if "target_phishing" in data.columns else 0,
        "jumlah_domain_unik": int(data["domain"].nunique()) if "domain" in data.columns else 0,
        "dataset": data["dataset_name"].value_counts().to_dict() if "dataset_name" in data.columns else {},
        "label_asli": data["original_label"].value_counts().head(30).to_dict() if "original_label" in data.columns else {},
    }

    lokasi = direktori_outputs / nama_file
    pd.DataFrame([{
        "jumlah_data": ringkasan["jumlah_data"],
        "jumlah_aman": ringkasan["jumlah_aman"],
        "jumlah_berisiko": ringkasan["jumlah_berisiko"],
        "jumlah_domain_unik": ringkasan["jumlah_domain_unik"],
        "dataset_json": json.dumps(ringkasan["dataset"], ensure_ascii=False),
        "label_asli_json": json.dumps(ringkasan["label_asli"], ensure_ascii=False),
    }]).to_csv(lokasi, index=False)

    print("Ringkasan disimpan:", lokasi)
    return ringkasan

## Preview Schema PhreshPhish

In [4]:
from datasets import load_dataset

print("Membaca preview PhreshPhish...")

preview_phresh = load_dataset(
    "phreshphish/phreshphish",
    split="train",
    streaming=True,
)

baris_preview = []

for i, row in enumerate(preview_phresh):
    if i >= 5:
        break

    baris_preview.append({
        "kolom_tersedia": list(row.keys()),
        "url": row.get("url", ""),
        "label": row.get("label", ""),
    })

data_preview_phresh = pd.DataFrame(baris_preview)

print("Preview PhreshPhish:")
display(data_preview_phresh)

Membaca preview PhreshPhish...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Preview PhreshPhish:


,kolom_tersedia,url,label
0,"[sha256, url, label, target, date, lang, lang_...",https://www.lifeonthemediterranean.com/portofi...,benign
1,"[sha256, url, label, target, date, lang, lang_...",https://moneypuck.com/goalies.htm,benign
2,"[sha256, url, label, target, date, lang, lang_...",https://learn.microsoft.com/en-us/azure/defend...,benign
3,"[sha256, url, label, target, date, lang, lang_...",https://www.grandcanyondestinations.com/las-ve...,benign
4,"[sha256, url, label, target, date, lang, lang_...",https://pgcps-md.safeschools.com/training/laun...,benign


## Ambil PhreshPhish URL + Label

In [5]:
import gc
import os
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

BATAS_PHRESHPHISH_TRAIN = 200_000
BATAS_PHRESHPHISH_TEST = 80_000
UKURAN_CHUNK_PHRESH = 5_000


def load_phreshphish_stream_ringan(split):
    """
    Load PhreshPhish hanya kolom url dan label.
    Cara utama memakai columns agar Parquet tidak membaca HTML/content besar.
    """
    try:
        dataset_stream = load_dataset(
            "phreshphish/phreshphish",
            split=split,
            streaming=True,
            columns=["url", "label"],
        )
        print(f"Load PhreshPhish {split} berhasil dengan column projection.")
        return dataset_stream

    except TypeError as error:
        print("Parameter columns belum didukung oleh versi datasets ini.")
        print("Mencoba fallback select_columns.")
        print("Pesan:", str(error)[:300])

        dataset_stream = load_dataset(
            "phreshphish/phreshphish",
            split=split,
            streaming=True,
        )

        try:
            dataset_stream = dataset_stream.select_columns(["url", "label"])
            print(f"Load PhreshPhish {split} berhasil dengan select_columns.")
            return dataset_stream
        except Exception as error_select:
            raise RuntimeError(
                "Gagal melakukan column projection. "
                "Update package datasets/pyarrow atau gunakan cell fallback Parquet manual."
            ) from error_select


def tulis_chunk_ke_csv(data_chunk, lokasi_output, tulis_header):
    data_chunk = pd.DataFrame(data_chunk)

    if data_chunk.empty:
        return 0

    data_standar = buat_dataset_standar(
        data=data_chunk,
        nama_dataset="PhreshPhish",
        sumber_data="huggingface_phreshphish",
        split=data_chunk["split_asli"].iloc[0] if "split_asli" in data_chunk.columns else "unknown",
    )

    data_standar.to_csv(
        lokasi_output,
        mode="a",
        index=False,
        header=tulis_header,
        encoding="utf-8",
    )

    jumlah = len(data_standar)

    del data_chunk
    del data_standar
    gc.collect()

    return jumlah


def ambil_phreshphish_split_memory_safe(split, batas_data, lokasi_output):
    print("=" * 70)
    print(f"Mengambil PhreshPhish split: {split}")
    print("Mode: memory safe, kolom dipakai hanya url dan label")
    print("Batas data:", batas_data)
    print("Output:", lokasi_output)

    lokasi_output = Path(lokasi_output)

    if lokasi_output.exists():
        lokasi_output.unlink()
        print("File output lama dihapus agar tidak tercampur:", lokasi_output)

    dataset_stream = load_phreshphish_stream_ringan(split)

    chunk = []
    total_tersimpan = 0
    tulis_header = True

    progress = tqdm(total=batas_data, desc=f"PhreshPhish {split}")

    for i, row in enumerate(dataset_stream):
        if i >= batas_data:
            break

        chunk.append({
            "url": row.get("url", ""),
            "label": row.get("label", ""),
            "split_asli": split,
        })

        if len(chunk) >= UKURAN_CHUNK_PHRESH:
            jumlah = tulis_chunk_ke_csv(chunk, lokasi_output, tulis_header)
            total_tersimpan += jumlah
            tulis_header = False
            chunk = []
            progress.update(UKURAN_CHUNK_PHRESH)

    if chunk:
        jumlah = tulis_chunk_ke_csv(chunk, lokasi_output, tulis_header)
        total_tersimpan += jumlah
        progress.update(len(chunk))
        chunk = []

    progress.close()
    gc.collect()

    if not lokasi_output.exists():
        raise RuntimeError(f"File output tidak terbentuk: {lokasi_output}")

    data_final = pd.read_csv(lokasi_output)
    data_final = data_final.drop_duplicates(subset=["url"]).reset_index(drop=True)
    data_final.to_csv(lokasi_output, index=False, encoding="utf-8")

    print(f"PhreshPhish {split} selesai.")
    print("Total tersimpan setelah deduplicate:", len(data_final))
    print("Distribusi target:")
    display(data_final["target_phishing"].value_counts())

    print("Distribusi label asli:")
    display(data_final["original_label"].value_counts().head(20))

    display(data_final.head(10))

    return data_final


lokasi_phresh_train = direktori_phreshphish / "phreshphish_train_url_label_sample.csv"
lokasi_phresh_test = direktori_phreshphish / "phreshphish_test_url_label_sample.csv"

data_phresh_train = ambil_phreshphish_split_memory_safe(
    split="train",
    batas_data=BATAS_PHRESHPHISH_TRAIN,
    lokasi_output=lokasi_phresh_train,
)

data_phresh_test = ambil_phreshphish_split_memory_safe(
    split="test",
    batas_data=BATAS_PHRESHPHISH_TEST,
    lokasi_output=lokasi_phresh_test,
)

print("PhreshPhish train disimpan:", lokasi_phresh_train)
print("PhreshPhish test disimpan:", lokasi_phresh_test)

print("Ukuran train:", data_phresh_train.shape)
print("Ukuran test:", data_phresh_test.shape)

Mengambil PhreshPhish split: train
Mode: memory safe, kolom dipakai hanya url dan label
Batas data: 200000
Output: C:\Users\ASUS\PHISHING\data\raw\phreshphish\phreshphish_train_url_label_sample.csv


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

...

target_phishing
0    43367
1    36633
Name: count, dtype: int64

Distribusi label asli:


original_label
benign    43367
phish     36633
Name: count, dtype: int64

,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,tanggal_diproses
0,https://calculatodo.com/en/count-days-between-...,calculatodo.com,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
1,https://cooktopcove.com/2024/09/23/you-can-alw...,cooktopcove.com,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
2,https://my.lafarmbureau.com/insurance/home,my.lafarmbureau.com,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
3,https://html5.adsrvr.org/amhavjr/wnbjgdc/6wakh...,html5.adsrvr.org,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
4,https://elbow.co.uk/,elbow.co.uk,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
5,http://coolfocus.info/google.php,coolfocus.info,phish,1,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
6,https://ucion.zaolir.cfd/pageamazon,ucion.zaolir.cfd,phish,1,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
7,https://jiu-jitsuinbridport.blogspot.pt/?m=1,jiu-jitsuinbridport.blogspot.pt,phish,1,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
8,https://treser-bridge.pages.dev,treser-bridge.pages.dev,phish,1,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42
9,https://en.wikipedia.org/wiki/History_of_mathe...,en.wikipedia.org,benign,0,PhreshPhish,huggingface_phreshphish,test,2026-05-18 19:41:42


PhreshPhish train disimpan: C:\Users\ASUS\PHISHING\data\raw\phreshphish\phreshphish_train_url_label_sample.csv
PhreshPhish test disimpan: C:\Users\ASUS\PHISHING\data\raw\phreshphish\phreshphish_test_url_label_sample.csv
Ukuran train: (199999, 8)
Ukuran test: (80000, 8)


## Preview Schema DeepURLBench

In [6]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

print("Membaca preview DeepURLBench...")
print("=" * 70)


def load_deepurlbench_stream_preview():
    """
    Mencoba beberapa cara load DeepURLBench.
    Ini dibuat fleksibel karena dataset HuggingFace kadang memakai config berbeda.
    """
    percobaan = [
        {
            "nama": "config_urls_without_dns",
            "args": ("davanstrien/DeepURLBench", "urls_without_dns"),
            "kwargs": {
                "split": "train",
                "streaming": True,
            },
        },
        {
            "nama": "data_dir_urls_without_dns",
            "args": ("davanstrien/DeepURLBench",),
            "kwargs": {
                "data_dir": "urls_without_dns",
                "split": "train",
                "streaming": True,
            },
        },
        {
            "nama": "default_train",
            "args": ("davanstrien/DeepURLBench",),
            "kwargs": {
                "split": "train",
                "streaming": True,
            },
        },
    ]

    error_terakhir = None

    for item in percobaan:
        try:
            print("Mencoba cara load:", item["nama"])

            dataset_stream = load_dataset(
                *item["args"],
                **item["kwargs"],
            )

            print("Berhasil load dengan cara:", item["nama"])

            return dataset_stream, item["nama"]

        except Exception as error:
            error_terakhir = error
            print("Gagal:", item["nama"])
            print("Pesan error:", str(error)[:500])
            print("-" * 70)

    raise RuntimeError(
        f"Semua cara load DeepURLBench gagal. Error terakhir: {error_terakhir}"
    )


preview_deep_stream, cara_load_deepurl = load_deepurlbench_stream_preview()

baris_preview = []

for i, row in enumerate(preview_deep_stream):
    if i >= 10:
        break

    baris_preview.append({
        "nomor": i + 1,
        "kolom_tersedia": list(row.keys()),
        "url": row.get("url", ""),
        "label": row.get("label", ""),
        "first_seen": row.get("first_seen", ""),
    })

data_preview_deep = pd.DataFrame(baris_preview)

print("Cara load DeepURLBench yang berhasil:", cara_load_deepurl)
print("Jumlah preview:", len(data_preview_deep))

display(data_preview_deep)

# Simpan preview agar terdokumentasi
lokasi_preview_deepurlbench = direktori_outputs / "preview_deepurlbench_step13.csv"
data_preview_deep.to_csv(lokasi_preview_deepurlbench, index=False)

print("Preview DeepURLBench disimpan:")
print(lokasi_preview_deepurlbench)

# Validasi kolom penting
if data_preview_deep.empty:
    raise RuntimeError("Preview DeepURLBench kosong. Cek koneksi internet atau akses HuggingFace.")

kolom_pertama = data_preview_deep["kolom_tersedia"].iloc[0]

print("\nValidasi kolom:")
print("Kolom url tersedia:", "url" in kolom_pertama)
print("Kolom label tersedia:", "label" in kolom_pertama)
print("Kolom first_seen tersedia:", "first_seen" in kolom_pertama)

if "url" not in kolom_pertama:
    raise ValueError("Kolom url tidak ditemukan pada DeepURLBench.")

if "label" not in kolom_pertama:
    raise ValueError("Kolom label tidak ditemukan pada DeepURLBench.")

print("\nCELL 6 selesai. DeepURLBench bisa dilanjutkan ke CELL 7.")

Membaca preview DeepURLBench...
Mencoba cara load: config_urls_without_dns


README.md: 0.00B [00:00, ?B/s]

C:\Users\ASUS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\datasets--davanstrien--DeepURLBench. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Berhasil load dengan cara: config_urls_without_dns
Cara load DeepURLBench yang berhasil: config_urls_without_dns
Jumlah preview: 10


,nomor,kolom_tersedia,url,label,first_seen
0,1,"[url, label, first_seen]",http://babearchives.com,benign,2019-02-11 22:52:00
1,2,"[url, label, first_seen]",http://timeline.antunes.dev,benign,2021-09-29 14:54:27
2,3,"[url, label, first_seen]",http://3md6bdtvo62lvedxjxiu7f2jkzbwab.1qw2es2z...,phishing,2022-08-07 07:30:11
3,4,"[url, label, first_seen]",https://www36.55.56.89.30.50.26.92.68.80.36.64...,phishing,2021-11-20 16:51:02
4,5,"[url, label, first_seen]",http://271h0ir.xcoysf.cn/adidas-me/css,phishing,2021-07-01 13:44:19
5,6,"[url, label, first_seen]",http://www.handsonintl.org/inkleet,phishing,2022-05-11 18:10:11
6,7,"[url, label, first_seen]",http://www19.22.54.85.44.89.59.35.90.52.35.67....,phishing,2021-11-23 05:51:28
7,8,"[url, label, first_seen]",http://nvoxnr7rksq.1qw2es2z.xyz,phishing,2022-08-01 06:00:13
8,9,"[url, label, first_seen]",http://mqamcj.islandspeechwrong.top,malware,2021-10-18 15:57:53
9,10,"[url, label, first_seen]",https://qua.rt-redirect.com/sl/dbb4ac31,benign,2022-04-15 16:45:17


Preview DeepURLBench disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\preview_deepurlbench_step13.csv

Validasi kolom:
Kolom url tersedia: True
Kolom label tersedia: True
Kolom first_seen tersedia: True

CELL 6 selesai. DeepURLBench bisa dilanjutkan ke CELL 7.


## Ambil DeepURLBench URL + Label

In [7]:
import gc
import os
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

TARGET_PER_LABEL_DEEPURL = {
    "benign": 120_000,
    "phishing": 120_000,
    "malware": 120_000,
}

BATAS_DEEPURLBENCH_SCAN = 1_500_000

UKURAN_CHUNK_DEEPURL = 10_000


def normalisasi_label_deepurl(label):
    """
    Menyamakan label DeepURLBench agar rapi.
    Kemungkinan label:
    - benign
    - phishing
    - mal
    - malware
    - malicious
    """
    teks = str(label).strip().lower()

    if teks in ["benign", "legit", "legitimate", "safe", "clean"]:
        return "benign"

    if teks in ["phishing", "phish"]:
        return "phishing"

    if teks in ["mal", "malware", "malicious"]:
        return "malware"

    return ""


def load_deepurlbench_stream_safe():
    """
    Mencoba beberapa cara load DeepURLBench agar aman di berbagai versi datasets.
    """
    percobaan = [
        {
            "nama": "config_urls_without_dns",
            "args": ("davanstrien/DeepURLBench", "urls_without_dns"),
            "kwargs": {
                "split": "train",
                "streaming": True,
            },
        },
        {
            "nama": "data_dir_urls_without_dns",
            "args": ("davanstrien/DeepURLBench",),
            "kwargs": {
                "data_dir": "urls_without_dns",
                "split": "train",
                "streaming": True,
            },
        },
        {
            "nama": "default_train",
            "args": ("davanstrien/DeepURLBench",),
            "kwargs": {
                "split": "train",
                "streaming": True,
            },
        },
    ]

    error_terakhir = None

    for item in percobaan:
        try:
            print("Mencoba load DeepURLBench:", item["nama"])

            dataset_stream = load_dataset(
                *item["args"],
                **item["kwargs"],
            )

            print("Berhasil load DeepURLBench dengan cara:", item["nama"])
            return dataset_stream, item["nama"]

        except Exception as error:
            error_terakhir = error
            print("Gagal:", item["nama"])
            print("Pesan:", str(error)[:500])
            print("-" * 70)

    raise RuntimeError(
        f"Semua cara load DeepURLBench gagal. Error terakhir: {error_terakhir}"
    )


def tulis_chunk_deepurl_ke_csv(data_chunk, lokasi_output, tulis_header):
    """
    Menulis chunk DeepURLBench ke CSV setelah distandarkan.
    """
    if not data_chunk:
        return 0

    data_chunk = pd.DataFrame(data_chunk)

    if data_chunk.empty:
        return 0

    data_standar = buat_dataset_standar(
        data=data_chunk,
        nama_dataset="DeepURLBench",
        sumber_data="huggingface_deepurlbench_urls_without_dns",
        split="train",
    )

    data_standar.to_csv(
        lokasi_output,
        mode="a",
        index=False,
        header=tulis_header,
        encoding="utf-8",
    )

    jumlah = len(data_standar)

    del data_chunk
    del data_standar
    gc.collect()

    return jumlah


def target_sudah_terpenuhi(hitung_label):
    for label, target in TARGET_PER_LABEL_DEEPURL.items():
        if hitung_label.get(label, 0) < target:
            return False

    return True


def ambil_deepurlbench_stream_memory_safe():
    print("=" * 70)
    print("Mengambil DeepURLBench subset urls_without_dns")
    print("Mode: streaming + chunk + label balancing")
    print("Target per label:", TARGET_PER_LABEL_DEEPURL)
    print("Batas scan:", BATAS_DEEPURLBENCH_SCAN)

    lokasi_output = direktori_deepurlbench / "deepurlbench_urls_without_dns_sample.csv"

    if lokasi_output.exists():
        lokasi_output.unlink()
        print("File output lama dihapus agar tidak tercampur:", lokasi_output)

    dataset_stream, cara_load = load_deepurlbench_stream_safe()

    hitung_label = {
        "benign": 0,
        "phishing": 0,
        "malware": 0,
    }

    chunk = []
    total_ditulis = 0
    tulis_header = True

    progress = tqdm(
        total=BATAS_DEEPURLBENCH_SCAN,
        desc="DeepURLBench scan",
    )

    for i, row in enumerate(dataset_stream):
        if i >= BATAS_DEEPURLBENCH_SCAN:
            break

        progress.update(1)

        label_asli = row.get("label", "")
        label_normal = normalisasi_label_deepurl(label_asli)

        if label_normal == "":
            continue

        if label_normal not in TARGET_PER_LABEL_DEEPURL:
            continue

        if hitung_label[label_normal] >= TARGET_PER_LABEL_DEEPURL[label_normal]:
            if target_sudah_terpenuhi(hitung_label):
                break
            continue

        url = row.get("url", "")

        chunk.append({
            "url": url,
            "label": label_normal,
            "original_label_raw": str(label_asli),
            "first_seen": row.get("first_seen", ""),
        })

        hitung_label[label_normal] += 1

        if len(chunk) >= UKURAN_CHUNK_DEEPURL:
            jumlah = tulis_chunk_deepurl_ke_csv(
                data_chunk=chunk,
                lokasi_output=lokasi_output,
                tulis_header=tulis_header,
            )

            total_ditulis += jumlah
            tulis_header = False
            chunk = []

            print("Progress label:", hitung_label)

        if target_sudah_terpenuhi(hitung_label):
            break

    if chunk:
        jumlah = tulis_chunk_deepurl_ke_csv(
            data_chunk=chunk,
            lokasi_output=lokasi_output,
            tulis_header=tulis_header,
        )

        total_ditulis += jumlah
        chunk = []

    progress.close()
    gc.collect()

    if not lokasi_output.exists():
        raise RuntimeError("File output DeepURLBench tidak terbentuk.")

    data_final = pd.read_csv(lokasi_output)

    sebelum_deduplikat = len(data_final)
    data_final = data_final.drop_duplicates(subset=["url"]).reset_index(drop=True)
    sesudah_deduplikat = len(data_final)

    data_final.to_csv(lokasi_output, index=False, encoding="utf-8")

    print("=" * 70)
    print("DeepURLBench selesai.")
    print("Cara load:", cara_load)
    print("File output:", lokasi_output)
    print("Total ditulis sebelum deduplicate:", sebelum_deduplikat)
    print("Total setelah deduplicate:", sesudah_deduplikat)
    print("Jumlah duplikat dibuang:", sebelum_deduplikat - sesudah_deduplikat)
    print("Hitung label saat ambil data:", hitung_label)

    print("\nDistribusi target:")
    display(data_final["target_phishing"].value_counts())

    print("\nDistribusi label asli:")
    display(data_final["original_label"].value_counts())

    print("\nPreview data:")
    display(data_final.head(10))

    return data_final, lokasi_output


data_deepurl, lokasi_deepurl = ambil_deepurlbench_stream_memory_safe()

print("DeepURLBench disimpan:")
print(lokasi_deepurl)

print("Ukuran data DeepURLBench:", data_deepurl.shape)

Mengambil DeepURLBench subset urls_without_dns
Mode: streaming + chunk + label balancing
Target per label: {'benign': 120000, 'phishing': 120000, 'malware': 120000}
Batas scan: 1500000
Mencoba load DeepURLBench: config_urls_without_dns


Resolving data files:   0%|          | 0/108 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Berhasil load DeepURLBench dengan cara: config_urls_without_dns


DeepURLBench scan:   0%|          | 0/1500000 [00:00<?, ?it/s]

...


Distribusi label asli:


original_label
benign      120000
phishing    120000
malware     120000
Name: count, dtype: int64


Preview data:


,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,tanggal_diproses,first_seen
0,http://babearchives.com,babearchives.com,benign,0,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2019-02-11 22:52:00
1,http://timeline.antunes.dev,timeline.antunes.dev,benign,0,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2021-09-29 14:54:27
2,http://3md6bdtvo62lvedxjxiu7f2jkzbwab.1qw2es2z...,3md6bdtvo62lvedxjxiu7f2jkzbwab.1qw2es2z.xyz,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2022-08-07 07:30:11
3,https://www36.55.56.89.30.50.26.92.68.80.36.64...,www36.55.56.89.30.50.26.92.68.80.36.64.91.60.4...,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2021-11-20 16:51:02
4,http://271h0ir.xcoysf.cn/adidas-me/css,271h0ir.xcoysf.cn,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2021-07-01 13:44:19
5,http://www.handsonintl.org/inkleet,handsonintl.org,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2022-05-11 18:10:11
6,http://www19.22.54.85.44.89.59.35.90.52.35.67....,www19.22.54.85.44.89.59.35.90.52.35.67.20.49.4...,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2021-11-23 05:51:28
7,http://nvoxnr7rksq.1qw2es2z.xyz,nvoxnr7rksq.1qw2es2z.xyz,phishing,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2022-08-01 06:00:13
8,http://mqamcj.islandspeechwrong.top,mqamcj.islandspeechwrong.top,malware,1,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2021-10-18 15:57:53
9,https://qua.rt-redirect.com/sl/dbb4ac31,qua.rt-redirect.com,benign,0,DeepURLBench,huggingface_deepurlbench_urls_without_dns,train,2026-05-18 19:56:35,2022-04-15 16:45:17


DeepURLBench disimpan:
C:\Users\ASUS\PHISHING\data\raw\deepurlbench\deepurlbench_urls_without_dns_sample.csv
Ukuran data DeepURLBench: (360000, 9)


## Baca Tranco top-1m.csv

In [8]:
import pandas as pd
from pathlib import Path
from datetime import datetime

BATAS_TRANCO = 100_000

lokasi_tranco = direktori_raw / "top-1m.csv"

if not lokasi_tranco.exists():
    raise FileNotFoundError(f"File Tranco tidak ditemukan: {lokasi_tranco}")

print("File Tranco ditemukan:")
print(lokasi_tranco)


def baca_tranco_top_1m(lokasi_file, batas=100_000):
    lokasi_file = Path(lokasi_file)
    
    data_awal = pd.read_csv(
        lokasi_file,
        header=None,
        nrows=5,
        encoding="utf-8",
        on_bad_lines="skip",
    )

    print("Preview struktur awal Tranco:")
    display(data_awal)

    if data_awal.shape[1] >= 2:
        data = pd.read_csv(
            lokasi_file,
            header=None,
            names=["rank", "domain"],
            nrows=batas,
            encoding="utf-8",
            on_bad_lines="skip",
        )
    else:
        data = pd.read_csv(
            lokasi_file,
            nrows=batas,
            encoding="utf-8",
            on_bad_lines="skip",
        )

        kolom_domain = cari_kolom(data, ["domain", "Domain", "url", "URL"])

        if kolom_domain is None:
            raise ValueError(f"Kolom domain tidak ditemukan. Kolom tersedia: {list(data.columns)}")

        data = data[[kolom_domain]].copy()
        data["rank"] = range(1, len(data) + 1)
        data = data.rename(columns={kolom_domain: "domain"})

    data["domain"] = data["domain"].astype(str).str.strip().str.lower()
    data["domain"] = data["domain"].str.replace("www.", "", n=1, regex=False)
    data = data[data["domain"] != ""]
    data = data.dropna(subset=["domain"])
    data = data.drop_duplicates(subset=["domain"]).reset_index(drop=True)

    hasil = pd.DataFrame()
    hasil["url"] = "https://" + data["domain"]
    hasil["domain"] = data["domain"]
    hasil["original_label"] = "tranco_legitimate"
    hasil["target_phishing"] = 0
    hasil["dataset_name"] = "Tranco"
    hasil["sumber_data"] = "local_tranco_top_1m"
    hasil["split"] = "legitimate_seed"
    hasil["tanggal_diproses"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    hasil["rank"] = data["rank"]

    hasil = hasil.drop_duplicates(subset=["url"]).reset_index(drop=True)

    return hasil


data_tranco = baca_tranco_top_1m(
    lokasi_file=lokasi_tranco,
    batas=BATAS_TRANCO,
)

lokasi_tranco_standar = direktori_tranco / "tranco_legitimate_url_seed.csv"
data_tranco.to_csv(lokasi_tranco_standar, index=False, encoding="utf-8")

print("=" * 70)
print("Tranco berhasil distandarkan.")
print("File sumber:", lokasi_tranco)
print("File output:", lokasi_tranco_standar)
print("Ukuran data Tranco:", data_tranco.shape)

print("\nDistribusi target:")
display(data_tranco["target_phishing"].value_counts())

print("\nDistribusi dataset:")
display(data_tranco["dataset_name"].value_counts())

print("\nPreview hasil Tranco:")
display(data_tranco.head(10))

File Tranco ditemukan:
C:\Users\ASUS\PHISHING\data\raw\top-1m.csv
Preview struktur awal Tranco:


,0,1
0,1,google.com
1,2,gtld-servers.net
2,3,cloudflare.com
3,4,gstatic.com
4,5,facebook.com


Tranco berhasil distandarkan.
File sumber: C:\Users\ASUS\PHISHING\data\raw\top-1m.csv
File output: C:\Users\ASUS\PHISHING\data\raw\tranco\tranco_legitimate_url_seed.csv
Ukuran data Tranco: (100000, 9)

Distribusi target:


target_phishing
0    100000
Name: count, dtype: int64


Distribusi dataset:


dataset_name
Tranco    100000
Name: count, dtype: int64


Preview hasil Tranco:


,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,tanggal_diproses,rank
0,https://google.com,google.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,1
1,https://gtld-servers.net,gtld-servers.net,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,2
2,https://cloudflare.com,cloudflare.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,3
3,https://gstatic.com,gstatic.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,4
4,https://facebook.com,facebook.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,5
5,https://microsoft.com,microsoft.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,6
6,https://googleapis.com,googleapis.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,7
7,https://youtube.com,youtube.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,8
8,https://amazonaws.com,amazonaws.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,9
9,https://apple.com,apple.com,tranco_legitimate,0,Tranco,local_tranco_top_1m,legitimate_seed,2026-05-18 20:04:06,10


## Menggabungkan PhreshPhish + DeepURLBench + Tranco

In [9]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 9 - Gabungkan PhreshPhish + DeepURLBench + Tranco")
print("=" * 80)

lokasi_phresh_train = direktori_phreshphish / "phreshphish_train_url_label_sample.csv"
lokasi_phresh_test = direktori_phreshphish / "phreshphish_test_url_label_sample.csv"
lokasi_deepurl = direktori_deepurlbench / "deepurlbench_urls_without_dns_sample.csv"
lokasi_tranco_standar = direktori_tranco / "tranco_legitimate_url_seed.csv"

file_input_wajib = [
    lokasi_phresh_train,
    lokasi_phresh_test,
    lokasi_deepurl,
    lokasi_tranco_standar,
]

for lokasi in file_input_wajib:
    if not lokasi.exists():
        raise FileNotFoundError(f"File wajib belum tersedia: {lokasi}")

print("Semua file input wajib tersedia.")

data_phresh_train = pd.read_csv(lokasi_phresh_train)
data_phresh_test = pd.read_csv(lokasi_phresh_test)
data_deepurl = pd.read_csv(lokasi_deepurl)
data_tranco = pd.read_csv(lokasi_tranco_standar)

print("Ukuran data PhreshPhish train:", data_phresh_train.shape)
print("Ukuran data PhreshPhish test :", data_phresh_test.shape)
print("Ukuran data DeepURLBench     :", data_deepurl.shape)
print("Ukuran data Tranco           :", data_tranco.shape)

kolom_wajib = [
    "url",
    "domain",
    "original_label",
    "target_phishing",
    "dataset_name",
    "sumber_data",
    "split",
    "tanggal_diproses",
]

def pastikan_kolom_wajib(data, nama_data):
    kolom_hilang = [kolom for kolom in kolom_wajib if kolom not in data.columns]

    if kolom_hilang:
        raise ValueError(f"{nama_data} kehilangan kolom wajib: {kolom_hilang}")

    return True


pastikan_kolom_wajib(data_phresh_train, "PhreshPhish train")
pastikan_kolom_wajib(data_phresh_test, "PhreshPhish test")
pastikan_kolom_wajib(data_deepurl, "DeepURLBench")
pastikan_kolom_wajib(data_tranco, "Tranco")

print("Semua dataset memiliki kolom wajib.")

data_tambahan_raw = pd.concat(
    [
        data_phresh_train,
        data_phresh_test,
        data_deepurl,
        data_tranco,
    ],
    ignore_index=True,
)

print("Ukuran gabungan mentah:", data_tambahan_raw.shape)

data_tambahan_raw["url"] = data_tambahan_raw["url"].astype(str).str.strip()
data_tambahan_raw["domain"] = data_tambahan_raw["domain"].astype(str).str.strip().str.lower()
data_tambahan_raw["original_label"] = data_tambahan_raw["original_label"].astype(str).str.strip().str.lower()
data_tambahan_raw["dataset_name"] = data_tambahan_raw["dataset_name"].astype(str).str.strip()
data_tambahan_raw["sumber_data"] = data_tambahan_raw["sumber_data"].astype(str).str.strip()
data_tambahan_raw["split"] = data_tambahan_raw["split"].astype(str).str.strip()

data_tambahan_raw = data_tambahan_raw[
    (data_tambahan_raw["url"] != "")
    & (data_tambahan_raw["domain"] != "")
].copy()

data_tambahan_raw["target_phishing"] = pd.to_numeric(
    data_tambahan_raw["target_phishing"],
    errors="coerce",
)

data_tambahan_raw = data_tambahan_raw.dropna(subset=["target_phishing"])
data_tambahan_raw["target_phishing"] = data_tambahan_raw["target_phishing"].astype(int)

data_tambahan_raw = data_tambahan_raw[
    data_tambahan_raw["target_phishing"].isin([0, 1])
].copy()

data_tambahan_raw = data_tambahan_raw.reset_index(drop=True)

print("Ukuran setelah cleaning dasar:", data_tambahan_raw.shape)

jumlah_target_per_url = (
    data_tambahan_raw
    .groupby("url")["target_phishing"]
    .nunique()
    .reset_index(name="jumlah_label_unik")
)

url_konflik = jumlah_target_per_url[
    jumlah_target_per_url["jumlah_label_unik"] > 1
]["url"].tolist()

data_konflik_url = data_tambahan_raw[
    data_tambahan_raw["url"].isin(url_konflik)
].sort_values(["url", "dataset_name", "target_phishing"])

lokasi_konflik_url = direktori_multi / "dataset_tambahan_konflik_label_url.csv"
data_konflik_url.to_csv(lokasi_konflik_url, index=False, encoding="utf-8")

print("Jumlah URL dengan konflik label:", len(url_konflik))
print("File konflik URL disimpan:", lokasi_konflik_url)

data_tambahan_tanpa_konflik = data_tambahan_raw[
    ~data_tambahan_raw["url"].isin(url_konflik)
].copy()

sebelum_deduplicate = len(data_tambahan_tanpa_konflik)

data_tambahan_clean = (
    data_tambahan_tanpa_konflik
    .drop_duplicates(subset=["url"])
    .reset_index(drop=True)
)

sesudah_deduplicate = len(data_tambahan_clean)

print("Jumlah sebelum deduplicate:", sebelum_deduplicate)
print("Jumlah setelah deduplicate :", sesudah_deduplicate)
print("Duplikat non-konflik dibuang:", sebelum_deduplicate - sesudah_deduplicate)

domain_audit = (
    data_tambahan_clean
    .groupby("domain")
    .agg(
        jumlah_data=("url", "count"),
        jumlah_label_unik=("target_phishing", "nunique"),
        jumlah_aman=("target_phishing", lambda x: int((x == 0).sum())),
        jumlah_berisiko=("target_phishing", lambda x: int((x == 1).sum())),
    )
    .reset_index()
)

data_konflik_domain = domain_audit[
    domain_audit["jumlah_label_unik"] > 1
].sort_values(["jumlah_data"], ascending=False)

lokasi_konflik_domain = direktori_multi / "dataset_tambahan_audit_konflik_domain.csv"
data_konflik_domain.to_csv(lokasi_konflik_domain, index=False, encoding="utf-8")

print("Jumlah domain dengan campuran label:", len(data_konflik_domain))
print("File audit domain disimpan:", lokasi_konflik_domain)

lokasi_dataset_tambahan_raw = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco.csv"
lokasi_dataset_tambahan_clean = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv"

data_tambahan_raw.to_csv(lokasi_dataset_tambahan_raw, index=False, encoding="utf-8")
data_tambahan_clean.to_csv(lokasi_dataset_tambahan_clean, index=False, encoding="utf-8")

print("\nDataset tambahan RAW disimpan:")
print(lokasi_dataset_tambahan_raw)

print("\nDataset tambahan CLEAN disimpan:")
print(lokasi_dataset_tambahan_clean)

print("\nDistribusi target CLEAN:")
display(data_tambahan_clean["target_phishing"].value_counts())

print("\nDistribusi dataset CLEAN:")
display(data_tambahan_clean["dataset_name"].value_counts())

print("\nDistribusi label asli CLEAN:")
display(data_tambahan_clean["original_label"].value_counts().head(30))

print("\nPreview dataset clean:")
display(data_tambahan_clean.head(10))

CELL 9 - Gabungkan PhreshPhish + DeepURLBench + Tranco
Semua file input wajib tersedia.
Ukuran data PhreshPhish train: (199999, 8)
Ukuran data PhreshPhish test : (80000, 8)
Ukuran data DeepURLBench     : (360000, 9)
Ukuran data Tranco           : (100000, 9)
Semua dataset memiliki kolom wajib.
Ukuran gabungan mentah: (739999, 10)
Ukuran setelah cleaning dasar: (739999, 10)
Jumlah URL dengan konflik label: 2
File konflik URL disimpan: C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_tambahan_konflik_label_url.csv
Jumlah sebelum deduplicate: 739995
Jumlah setelah deduplicate : 739961
Duplikat non-konflik dibuang: 34
Jumlah domain dengan campuran label: 480
File audit domain disimpan: C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_tambahan_audit_konflik_domain.csv

Dataset tambahan RAW disimpan:
C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_tambahan_phreshphish_deepurlbench_tranco.csv

Dataset tambahan CLEAN disimpan:
C:\Users\ASUS\PHISHING\data\proce

target_phishing
0    374234
1    365727
Name: count, dtype: int64


Distribusi dataset CLEAN:


dataset_name
DeepURLBench    359992
PhreshPhish     279999
Tranco           99970
Name: count, dtype: int64


Distribusi label asli CLEAN:


original_label
benign               274264
phish                125730
phishing             120000
malware              119997
tranco_legitimate     99970
Name: count, dtype: int64


Preview dataset clean:


,url,domain,original_label,target_phishing,dataset_name,sumber_data,split,tanggal_diproses,first_seen,rank
0,https://www.lifeonthemediterranean.com/portofi...,lifeonthemediterranean.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
1,https://moneypuck.com/goalies.htm,moneypuck.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
2,https://learn.microsoft.com/en-us/azure/defend...,learn.microsoft.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
3,https://www.grandcanyondestinations.com/las-ve...,grandcanyondestinations.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
4,https://pgcps-md.safeschools.com/training/laun...,pgcps-md.safeschools.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
5,screenrant.com/orville-show-why-halston-sage-l...,screenrant.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
6,https://blog.hobartcorp.com/blog/when-should-y...,blog.hobartcorp.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
7,https://www-thediamondempirellc-com.filesusr.c...,www-thediamondempirellc-com.filesusr.com,phish,1,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
8,https://user-confirmation.fanpages-improve3658...,user-confirmation.fanpages-improve365823.com,phish,1,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN
9,https://www.unitedsiteservices.com/billpay/,unitedsiteservices.com,benign,0,PhreshPhish,huggingface_phreshphish,train,2026-05-18 19:40:16,NaN,NaN


## Validasi dan Audit Dataset Tambahan

In [10]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 10 - Validasi dan Audit Dataset Tambahan")
print("=" * 80)

lokasi_dataset_tambahan_raw = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco.csv"
lokasi_dataset_tambahan_clean = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv"
lokasi_konflik_url = direktori_multi / "dataset_tambahan_konflik_label_url.csv"
lokasi_konflik_domain = direktori_multi / "dataset_tambahan_audit_konflik_domain.csv"

file_validasi = [
    lokasi_phresh_train,
    lokasi_phresh_test,
    lokasi_deepurl,
    lokasi_tranco_standar,
    lokasi_dataset_tambahan_raw,
    lokasi_dataset_tambahan_clean,
    lokasi_konflik_url,
    lokasi_konflik_domain,
]

data_validasi_file = []

for lokasi in file_validasi:
    lokasi = Path(lokasi)
    data_validasi_file.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_mb": round(lokasi.stat().st_size / (1024 * 1024), 2) if lokasi.exists() else 0,
    })

data_validasi_file = pd.DataFrame(data_validasi_file)

print("Validasi file:")
display(data_validasi_file)

if not lokasi_dataset_tambahan_clean.exists():
    raise FileNotFoundError(f"Dataset clean belum tersedia: {lokasi_dataset_tambahan_clean}")

data_clean = pd.read_csv(lokasi_dataset_tambahan_clean)

print("Ukuran dataset clean:", data_clean.shape)

kolom_wajib_dataset_final = [
    "url",
    "domain",
    "original_label",
    "target_phishing",
    "dataset_name",
    "sumber_data",
    "split",
    "tanggal_diproses",
]

hasil_validasi_struktur = []

for kolom in kolom_wajib_dataset_final:
    hasil_validasi_struktur.append({
        "komponen": f"kolom_{kolom}",
        "status": kolom in data_clean.columns,
        "catatan": "tersedia" if kolom in data_clean.columns else "tidak tersedia",
    })

data_validasi_struktur = pd.DataFrame(hasil_validasi_struktur)

print("Validasi struktur:")
display(data_validasi_struktur)

jumlah_data = len(data_clean)
jumlah_url_unik = data_clean["url"].nunique()
jumlah_domain_unik = data_clean["domain"].nunique()
jumlah_null_url = int(data_clean["url"].isna().sum())
jumlah_null_domain = int(data_clean["domain"].isna().sum())
jumlah_target_tidak_valid = int((~data_clean["target_phishing"].isin([0, 1])).sum())
jumlah_duplikat_url = int(data_clean.duplicated(subset=["url"]).sum())

jumlah_aman = int((data_clean["target_phishing"] == 0).sum())
jumlah_berisiko = int((data_clean["target_phishing"] == 1).sum())

rasio_aman = round(jumlah_aman / jumlah_data, 4) if jumlah_data else 0
rasio_berisiko = round(jumlah_berisiko / jumlah_data, 4) if jumlah_data else 0

data_validasi_isi = pd.DataFrame([
    {"metrik": "jumlah_data", "nilai": jumlah_data},
    {"metrik": "jumlah_url_unik", "nilai": jumlah_url_unik},
    {"metrik": "jumlah_domain_unik", "nilai": jumlah_domain_unik},
    {"metrik": "jumlah_null_url", "nilai": jumlah_null_url},
    {"metrik": "jumlah_null_domain", "nilai": jumlah_null_domain},
    {"metrik": "jumlah_target_tidak_valid", "nilai": jumlah_target_tidak_valid},
    {"metrik": "jumlah_duplikat_url", "nilai": jumlah_duplikat_url},
    {"metrik": "jumlah_aman", "nilai": jumlah_aman},
    {"metrik": "jumlah_berisiko", "nilai": jumlah_berisiko},
    {"metrik": "rasio_aman", "nilai": rasio_aman},
    {"metrik": "rasio_berisiko", "nilai": rasio_berisiko},
])

print("Validasi isi:")
display(data_validasi_isi)

ringkasan_dataset = (
    data_clean
    .groupby(["dataset_name", "target_phishing"])
    .size()
    .reset_index(name="jumlah_data")
    .sort_values(["dataset_name", "target_phishing"])
)

print("Ringkasan dataset dan target:")
display(ringkasan_dataset)

ringkasan_label_asli = (
    data_clean["original_label"]
    .value_counts()
    .reset_index()
)

ringkasan_label_asli.columns = ["original_label", "jumlah_data"]

print("Ringkasan label asli:")
display(ringkasan_label_asli.head(30))

lokasi_validasi_file = direktori_outputs / "validasi_file_multi_dataset_collector_step13.csv"
lokasi_validasi_struktur = direktori_outputs / "validasi_struktur_multi_dataset_collector_step13.csv"
lokasi_validasi_isi = direktori_outputs / "validasi_isi_multi_dataset_collector_step13.csv"
lokasi_ringkasan_dataset = direktori_outputs / "ringkasan_dataset_tambahan_step13.csv"
lokasi_ringkasan_label = direktori_outputs / "ringkasan_label_dataset_tambahan_step13.csv"

data_validasi_file.to_csv(lokasi_validasi_file, index=False, encoding="utf-8")
data_validasi_struktur.to_csv(lokasi_validasi_struktur, index=False, encoding="utf-8")
data_validasi_isi.to_csv(lokasi_validasi_isi, index=False, encoding="utf-8")
ringkasan_dataset.to_csv(lokasi_ringkasan_dataset, index=False, encoding="utf-8")
ringkasan_label_asli.to_csv(lokasi_ringkasan_label, index=False, encoding="utf-8")

print("File validasi disimpan:")
print(lokasi_validasi_file)
print(lokasi_validasi_struktur)
print(lokasi_validasi_isi)
print(lokasi_ringkasan_dataset)
print(lokasi_ringkasan_label)

status_siap = True
catatan_status = []

if jumlah_data == 0:
    status_siap = False
    catatan_status.append("Dataset clean kosong.")

if jumlah_duplikat_url > 0:
    status_siap = False
    catatan_status.append("Masih ada duplikat URL.")

if jumlah_target_tidak_valid > 0:
    status_siap = False
    catatan_status.append("Masih ada target tidak valid.")

if jumlah_null_url > 0 or jumlah_null_domain > 0:
    status_siap = False
    catatan_status.append("Masih ada URL/domain kosong.")

if jumlah_aman == 0 or jumlah_berisiko == 0:
    status_siap = False
    catatan_status.append("Distribusi target tidak lengkap.")

if not catatan_status:
    catatan_status.append("Dataset tambahan bersih dan siap untuk standardisasi gabungan dengan PhiUSIIL.")

data_status_kesiapan = pd.DataFrame([{
    "status_siap_lanjut": status_siap,
    "jumlah_data": jumlah_data,
    "jumlah_aman": jumlah_aman,
    "jumlah_berisiko": jumlah_berisiko,
    "jumlah_domain_unik": jumlah_domain_unik,
    "catatan": " | ".join(catatan_status),
}])

lokasi_status_kesiapan = direktori_outputs / "status_kesiapan_dataset_tambahan_step13.csv"
data_status_kesiapan.to_csv(lokasi_status_kesiapan, index=False, encoding="utf-8")

print("\nStatus kesiapan:")
display(data_status_kesiapan)

print("Status kesiapan disimpan:")
print(lokasi_status_kesiapan)

CELL 10 - Validasi dan Audit Dataset Tambahan
Validasi file:


,nama_file,lokasi,tersedia,ukuran_mb
0,phreshphish_train_url_label_sample.csv,C:\Users\ASUS\PHISHING\data\raw\phreshphish\ph...,True,28.70
1,phreshphish_test_url_label_sample.csv,C:\Users\ASUS\PHISHING\data\raw\phreshphish\ph...,True,11.56
2,deepurlbench_urls_without_dns_sample.csv,C:\Users\ASUS\PHISHING\data\raw\deepurlbench\d...,True,68.20
3,tranco_legitimate_url_seed.csv,C:\Users\ASUS\PHISHING\data\raw\tranco\tranco_...,True,11.92
4,dataset_tambahan_phreshphish_deepurlbench_tran...,C:\Users\ASUS\PHISHING\data\processed\multi_da...,True,121.56
5,dataset_tambahan_phreshphish_deepurlbench_tran...,C:\Users\ASUS\PHISHING\data\processed\multi_da...,True,121.55
6,dataset_tambahan_konflik_label_url.csv,C:\Users\ASUS\PHISHING\data\processed\multi_da...,True,0.00
7,dataset_tambahan_audit_konflik_domain.csv,C:\Users\ASUS\PHISHING\data\processed\multi_da...,True,0.01


C:\Users\ASUS\AppData\Local\Temp\ipykernel_25404\4199099155.py:46: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  data_clean = pd.read_csv(lokasi_dataset_tambahan_clean)


Ukuran dataset clean: (739961, 10)
Validasi struktur:


,komponen,status,catatan
0,kolom_url,True,tersedia
1,kolom_domain,True,tersedia
2,kolom_original_label,True,tersedia
3,kolom_target_phishing,True,tersedia
4,kolom_dataset_name,True,tersedia
5,kolom_sumber_data,True,tersedia
6,kolom_split,True,tersedia
7,kolom_tanggal_diproses,True,tersedia


...

Ringkasan label asli:


,original_label,jumlah_data
0,benign,274264
1,phish,125730
2,phishing,120000
3,malware,119997
4,tranco_legitimate,99970


File validasi disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_file_multi_dataset_collector_step13.csv
C:\Users\ASUS\PHISHING\reports\outputs\validasi_struktur_multi_dataset_collector_step13.csv
C:\Users\ASUS\PHISHING\reports\outputs\validasi_isi_multi_dataset_collector_step13.csv
C:\Users\ASUS\PHISHING\reports\outputs\ringkasan_dataset_tambahan_step13.csv
C:\Users\ASUS\PHISHING\reports\outputs\ringkasan_label_dataset_tambahan_step13.csv

Status kesiapan:


,status_siap_lanjut,jumlah_data,jumlah_aman,jumlah_berisiko,jumlah_domain_unik,catatan
0,True,739961,374234,365727,571872,Dataset tambahan bersih dan siap untuk standar...


Status kesiapan disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\status_kesiapan_dataset_tambahan_step13.csv


## Metadata dan Catatan Final Collector

In [11]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd

print("=" * 80)
print("CELL 11 - Metadata dan Catatan Final Multi Dataset Collector")
print("=" * 80)

if "data_status_kesiapan" not in globals():
    lokasi_status_kesiapan = direktori_outputs / "status_kesiapan_dataset_tambahan_step13.csv"
    data_status_kesiapan = pd.read_csv(lokasi_status_kesiapan)

if "data_clean" not in globals():
    lokasi_dataset_tambahan_clean = direktori_multi / "dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv"
    data_clean = pd.read_csv(lokasi_dataset_tambahan_clean)

jumlah_data = int(len(data_clean))
jumlah_aman = int((data_clean["target_phishing"] == 0).sum())
jumlah_berisiko = int((data_clean["target_phishing"] == 1).sum())
jumlah_domain_unik = int(data_clean["domain"].nunique())

dataset_counts = data_clean["dataset_name"].value_counts().to_dict()
label_counts = data_clean["original_label"].value_counts().head(30).to_dict()

metadata_collector = {
    "nama_notebook": "13_multi_dataset_collector.ipynb",
    "nama_tahap_program": "STEP 14 - Multi Dataset Collector & Standardization Preparation",
    "status": "selesai",
    "tujuan": "Mengumpulkan dan menstandarkan dataset tambahan untuk persiapan retraining model PhishRisk V5.",
    "dataset_digunakan": [
        "PhreshPhish",
        "DeepURLBench",
        "Tranco",
    ],
    "file_input": {
        "phreshphish_train": str(lokasi_phresh_train),
        "phreshphish_test": str(lokasi_phresh_test),
        "deepurlbench": str(lokasi_deepurl),
        "tranco": str(lokasi_tranco_standar),
    },
    "file_output": {
        "dataset_tambahan_raw": str(lokasi_dataset_tambahan_raw),
        "dataset_tambahan_clean": str(lokasi_dataset_tambahan_clean),
        "konflik_label_url": str(lokasi_konflik_url),
        "audit_konflik_domain": str(lokasi_konflik_domain),
        "validasi_file": str(lokasi_validasi_file),
        "validasi_struktur": str(lokasi_validasi_struktur),
        "validasi_isi": str(lokasi_validasi_isi),
        "ringkasan_dataset": str(lokasi_ringkasan_dataset),
        "ringkasan_label": str(lokasi_ringkasan_label),
        "status_kesiapan": str(lokasi_status_kesiapan),
    },
    "ringkasan": {
        "jumlah_data_clean": jumlah_data,
        "jumlah_aman": jumlah_aman,
        "jumlah_berisiko": jumlah_berisiko,
        "jumlah_domain_unik": jumlah_domain_unik,
        "distribusi_dataset": dataset_counts,
        "distribusi_label_asli": label_counts,
    },
    "catatan_teknis": [
        "PhreshPhish diambil hanya kolom URL dan label agar tidak membebani RAM dengan HTML besar.",
        "DeepURLBench memakai subset urls_without_dns agar cocok dengan model URL-only.",
        "Tranco dipakai sebagai legitimate seed untuk menekan false positive.",
        "URL konflik label disimpan terpisah dan tidak dipakai di dataset clean.",
        "Konflik domain hanya diaudit karena satu domain bisa memiliki halaman aman dan halaman berisiko.",
        "Dataset ini belum langsung dipakai training. Tahap berikutnya adalah integrasi dengan PhiUSIIL dan feature extraction V5.",
    ],
    "tanggal_selesai": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

lokasi_metadata_collector = direktori_outputs / "metadata_multi_dataset_collector_step13.json"
lokasi_metadata_collector.write_text(
    json.dumps(metadata_collector, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

catatan_final = f"""
CATATAN FINAL MULTI DATASET COLLECTOR
=====================================

Nama notebook:
13_multi_dataset_collector.ipynb

Nama tahap program:
STEP 14 - Multi Dataset Collector & Standardization Preparation

Tujuan:
Mengumpulkan dataset tambahan untuk memperkuat PhishRisk agar tidak hanya bergantung pada satu dataset.

Dataset yang berhasil dipakai:
1. PhreshPhish
2. DeepURLBench
3. Tranco

Output utama:
{lokasi_dataset_tambahan_clean}

Ringkasan dataset clean:
- Jumlah data          : {jumlah_data}
- Jumlah aman          : {jumlah_aman}
- Jumlah berisiko      : {jumlah_berisiko}
- Jumlah domain unik   : {jumlah_domain_unik}

Distribusi dataset:
{json.dumps(dataset_counts, indent=4, ensure_ascii=False)}

Distribusi label asli:
{json.dumps(label_counts, indent=4, ensure_ascii=False)}

Catatan penting:
- PhreshPhish hanya diambil URL dan label.
- DeepURLBench memakai urls_without_dns.
- Tranco dipakai sebagai legitimate seed.
- URL dengan konflik label tidak dimasukkan ke dataset clean.
- Domain campuran label hanya diaudit, tidak langsung dihapus.
- Dataset tambahan ini siap untuk tahap berikutnya: integrasi dengan PhiUSIIL dan retraining model V5.

Tahap berikutnya yang direkomendasikan:
14_feature_extraction_multi_dataset_v5.ipynb

Isi tahap berikutnya:
1. Gabungkan PhiUSIIL + dataset tambahan clean.
2. Samakan struktur kolom URL dan target.
3. Ekstrak fitur URL manual.
4. Ekstrak fitur intelligence.
5. Buat dataset training V5.
6. Training model V5.
7. Evaluasi multi-dataset.
8. Bandingkan V2 vs V5.
"""

lokasi_catatan_final = direktori_outputs / "catatan_final_multi_dataset_collector_step13.txt"
lokasi_catatan_final.write_text(catatan_final, encoding="utf-8")

print(catatan_final)
print("Metadata collector disimpan:")
print(lokasi_metadata_collector)
print("Catatan final disimpan:")
print(lokasi_catatan_final)

CELL 11 - Metadata dan Catatan Final Multi Dataset Collector

CATATAN FINAL MULTI DATASET COLLECTOR

Nama notebook:
13_multi_dataset_collector.ipynb

Nama tahap program:
STEP 14 - Multi Dataset Collector & Standardization Preparation

Tujuan:
Mengumpulkan dataset tambahan untuk memperkuat PhishRisk agar tidak hanya bergantung pada satu dataset.

Dataset yang berhasil dipakai:
1. PhreshPhish
2. DeepURLBench
3. Tranco

Output utama:
C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv

Ringkasan dataset clean:
- Jumlah data          : 739961
- Jumlah aman          : 374234
- Jumlah berisiko      : 365727
- Jumlah domain unik   : 571872

Distribusi dataset:
{
    "DeepURLBench": 359992,
    "PhreshPhish": 279999,
    "Tranco": 99970
}

Distribusi label asli:
{
    "benign": 274264,
    "phish": 125730,
    "phishing": 120000,
    "malware": 119997,
    "tranco_legitimate": 99970
}

Catatan penting:
- PhreshPhish hanya diambil URL d

## Ringkasan Akhir

In [12]:
print("=" * 80)
print("RINGKASAN AKHIR - 13_multi_dataset_collector.ipynb")
print("=" * 80)

file_akhir = [
    lokasi_dataset_tambahan_raw,
    lokasi_dataset_tambahan_clean,
    lokasi_konflik_url,
    lokasi_konflik_domain,
    lokasi_validasi_file,
    lokasi_validasi_struktur,
    lokasi_validasi_isi,
    lokasi_ringkasan_dataset,
    lokasi_ringkasan_label,
    lokasi_status_kesiapan,
    lokasi_metadata_collector,
    lokasi_catatan_final,
]

ringkasan_file_akhir = []

for lokasi in file_akhir:
    lokasi = Path(lokasi)
    ringkasan_file_akhir.append({
        "nama_file": lokasi.name,
        "tersedia": lokasi.exists(),
        "ukuran_mb": round(lokasi.stat().st_size / (1024 * 1024), 2) if lokasi.exists() else 0,
        "lokasi": str(lokasi),
    })

ringkasan_file_akhir = pd.DataFrame(ringkasan_file_akhir)

display(ringkasan_file_akhir)

print("\nDataset clean:")
print(lokasi_dataset_tambahan_clean)

print("\nStatus kesiapan:")
display(data_status_kesiapan)

print("\nDistribusi dataset clean:")
display(data_clean["dataset_name"].value_counts())

print("\nDistribusi target clean:")
display(data_clean["target_phishing"].value_counts())

print("\nSelesai. Dataset tambahan siap untuk integrasi dengan PhiUSIIL dan retraining model V5.")

RINGKASAN AKHIR - 13_multi_dataset_collector.ipynb


,nama_file,tersedia,ukuran_mb,lokasi
0,dataset_tambahan_phreshphish_deepurlbench_tran...,True,121.56,C:\Users\ASUS\PHISHING\data\processed\multi_da...
1,dataset_tambahan_phreshphish_deepurlbench_tran...,True,121.55,C:\Users\ASUS\PHISHING\data\processed\multi_da...
2,dataset_tambahan_konflik_label_url.csv,True,0.00,C:\Users\ASUS\PHISHING\data\processed\multi_da...
3,dataset_tambahan_audit_konflik_domain.csv,True,0.01,C:\Users\ASUS\PHISHING\data\processed\multi_da...
4,validasi_file_multi_dataset_collector_step13.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\validas...
5,validasi_struktur_multi_dataset_collector_step...,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\validas...
6,validasi_isi_multi_dataset_collector_step13.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\validas...
7,ringkasan_dataset_tambahan_step13.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\ringkas...
8,ringkasan_label_dataset_tambahan_step13.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\ringkas...
9,status_kesiapan_dataset_tambahan_step13.csv,True,0.00,C:\Users\ASUS\PHISHING\reports\outputs\status_...



Dataset clean:
C:\Users\ASUS\PHISHING\data\processed\multi_dataset\dataset_tambahan_phreshphish_deepurlbench_tranco_clean.csv

Status kesiapan:


,status_siap_lanjut,jumlah_data,jumlah_aman,jumlah_berisiko,jumlah_domain_unik,catatan
0,True,739961,374234,365727,571872,Dataset tambahan bersih dan siap untuk standar...



Distribusi dataset clean:


dataset_name
DeepURLBench    359992
PhreshPhish     279999
Tranco           99970
Name: count, dtype: int64


Distribusi target clean:


target_phishing
0    374234
1    365727
Name: count, dtype: int64


Selesai. Dataset tambahan siap untuk integrasi dengan PhiUSIIL dan retraining model V5.
